In [1]:
import os
import re
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import harmonypy as hm
import matplotlib as mpl
import matplotlib.pyplot as plt 
from pathlib import Path
import anndata as ad

# Optional Harmony
HAVE_HARMONY = True

# Plot defaults
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=1200, facecolor='white', fontsize=8)
mpl.rcParams['savefig.dpi'] = 1200
mpl.rcParams['savefig.bbox'] = 'tight'

# --------------------
# Parameters
# --------------------
ROOT = "/group/canc2/anson/working/cf-pbmc-bal"

CONDITION_KEY = "Condition2"
BATCH_KEY = "sampleID"
CLUSTER_KEY = "ann_lv2"

COUNTS_LAYER = "counts"

N_HVG = 2000
N_PCS = 20
N_NEIGHBORS = 15

OUTDIR = os.path.join(ROOT,"data","plots","module_scoring")
os.makedirs(OUTDIR, exist_ok=True)
sc.settings.figdir = OUTDIR

def fraction_non_integer_like(X, tol=1e-6, sample_n=200000, seed=0):
    if sp.issparse(X):
        vals = X.data
    else:
        vals = np.asarray(X).ravel()
    if vals.size == 0:
        return 0.0
    rng = np.random.default_rng(seed)
    if vals.size > sample_n:
        vals = vals[rng.choice(vals.size, size=sample_n, replace=False)]
    return float(np.mean(np.abs(vals - np.round(vals)) > tol))


def ensure_counts_layer_from_X(adata: sc.AnnData, layer_key: str = COUNTS_LAYER, tol=1e-6, max_frac_nonint=1e-4):
    if layer_key in adata.layers:
        return
    frac = fraction_non_integer_like(adata.X, tol=tol)
    print(f"fraction non-integer-like in adata.X: {frac:.6g}")
    if frac > max_frac_nonint:
        raise ValueError(
            f"adata.X is not integer-like enough to treat as raw counts (frac_nonint={frac}).\n"
            "Re-convert/export with counts preserved is recommended."
        )
    if sp.issparse(adata.X):
        Xc = adata.X.copy()
        Xc.data = np.rint(Xc.data).astype(np.int32)
        adata.layers[layer_key] = Xc
    else:
        adata.layers[layer_key] = np.rint(np.asarray(adata.X)).astype(np.int32)


def set_counts_as_X(adata: sc.AnnData, layer_key: str = COUNTS_LAYER):
    if layer_key not in adata.layers:
        raise KeyError(f"Missing layer '{layer_key}'")
    adata.X = adata.layers[layer_key].copy()
    return adata


In [2]:
ROOT = "/group/canc2/anson/working/cf-pbmc-bal"
PBMC_H5AD = os.path.join(ROOT, "data", "SCEs", "analysis", "pbmc.h5ad")
BAL_H5AD  = os.path.join(ROOT, "data", "SCEs", "analysis", "bal.h5ad")

COUNTS_LAYER = "counts"

adata_pbmc = sc.read_h5ad(PBMC_H5AD)
adata_bal = sc.read_h5ad(BAL_H5AD)

for name, ad in [("pbmc", adata_pbmc), ("bal", adata_bal)]:
    for k in [CONDITION_KEY, BATCH_KEY, CLUSTER_KEY]:
        if k not in ad.obs.columns:
            raise KeyError(f"[{name}] Expected obs column '{k}' not found.")
    print(f"{name}: shape={ad.shape} layers={list(ad.layers.keys())}")
    print(f"{name}: X min/max={float(ad.X.min())} {float(ad.X.max())}")

# Keep counts layer + set as X for concatenation
print("Ensuring counts layers...")
ensure_counts_layer_from_X(adata_pbmc, layer_key=COUNTS_LAYER)
ensure_counts_layer_from_X(adata_bal, layer_key=COUNTS_LAYER)
print("pbmc layers:", list(adata_pbmc.layers.keys()))
print("bal layers:", list(adata_bal.layers.keys()))

set_counts_as_X(adata_pbmc, COUNTS_LAYER)
set_counts_as_X(adata_bal, COUNTS_LAYER)

# Concatenate (mapping keys already define dataset labels; do NOT pass keys=...)
adata = sc.concat(
    {"PBMC": adata_pbmc, "BAL": adata_bal},
    join="outer",
    label="dataset",
    merge="same"
)

# Make obs names unique (avoids the warning you saw)
adata.obs_names_make_unique()

# Ensure counts layer exists post-concat
if COUNTS_LAYER not in adata.layers:
    # If concat dropped layers, reconstruct counts from X (should still be counts)
    ensure_counts_layer_from_X(adata, layer_key=COUNTS_LAYER)

# Set ann_lv2 as categorical with an automatically derived, stable ordering:
# PBMC_* first, then BAL_*; otherwise alphabetical.
levels = sorted(adata.obs[CLUSTER_KEY].astype(str).unique().tolist())
levels = sorted(levels, key=lambda s: (0 if s.startswith("PBMC_") else 1, s))
adata.obs[CLUSTER_KEY] = pd.Categorical(adata.obs[CLUSTER_KEY].astype(str), categories=levels, ordered=True)

sc.pp.normalize_total(adata, target_sum = 1e4)
sc.pp.log1p(adata)
adata.layers["log1p"] = adata.X.copy()

pbmc: shape=(110809, 31166) layers=[]
pbmc: X min/max=0.0 17935.0
bal: shape=(107598, 31717) layers=[]
bal: X min/max=0.0 6411.0
Ensuring counts layers...
fraction non-integer-like in adata.X: 0
fraction non-integer-like in adata.X: 0
pbmc layers: ['counts']
bal layers: ['counts']


/group/canc2/anson/miniconda3/envs/python310/lib/python3.10/site-packages/anndata/_core/anndata.py:1756: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


normalizing counts per cell
    finished (0:00:01)


In [3]:
adata

AnnData object with n_obs × n_vars = 218407 × 32566
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'Barcode', 'Capture', 'HTODemux.ID', 'HTODemux_result.orig.ident', 'HTODemux_result.nCount_RNA', 'HTODemux_result.nFeature_RNA', 'HTODemux_result.nCount_HTO', 'HTODemux_result.nFeature_HTO', 'HTODemux_result.HTO_maxID', 'HTODemux_result.HTO_secondID', 'HTODemux_result.HTO_margin', 'HTODemux_result.HTO_classification', 'HTODemux_result.HTO_classification.global', 'HTODemux_result.hash.ID', 'sampleID.HTO', 'colours.hto_colours', 'colours.sample_colours', 'colours.capture_colours', 'colours.genetic_donor_colours', 'colours.sampleID.HTO_colours', 'colours.sampleID.genetics_colours', 'vireo.donor_id', 'vireo.prob_max', 'vireo.prob_doublet', 'vireo.n_vars', 'vireo.best_singlet', 'vireo.best_doublet', 'vireo.doublet_logLikRatio', 'genetic_donor', 'sampleID.genetics', 'sampleID', 'Age', 'Sex', 'Condition', 'Bronchiectasis', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'su

### Prepare gene sets and datasets

In [4]:
TNF_genes = ["ABCA1","ACKR3","AREG","ATF3","ATP2B1","B4GALT1","B4GALT5","BCL2A1","BCL3","BCL6","BHLHE40","BIRC2","BIRC3","BMP2","BTG1","BTG2","BTG3","CCL2","CCL20","CCL4","CCL5","CCN1","CCND1","CCNL1","CCRL2","CD44","CD69","CD80","CD83","CDKN1A","CEBPB","CEBPD","CFLAR","CLCF1","CSF1","CSF2","CXCL1","CXCL10","CXCL11","CXCL2","CXCL3","CXCL6","DDX58","DENND5A","DNAJB4","DRAM1","DUSP1","DUSP2","DUSP4","DUSP5","EDN1","EFNA1","EGR1","EGR2","EGR3","EHD1","EIF1","ETS2","F2RL1","F3","FJX1","FOS","FOSB","FOSL1","FOSL2","FUT4","G0S2","GADD45A","GADD45B","GCH1","GEM","GFPT2","GPR183","HBEGF","HES1","ICAM1","ICOSLG","ID2","IER2","IER3","IER5","IFIH1","IFIT2","IFNGR2","IL12B","IL15RA","IL18","IL1A","IL1B","IL23A","IL6","IL6ST","IL7R","INHBA","IRF1","IRS2","JAG1","JUN","JUNB","KDM6B","KLF10","KLF2","KLF4","KLF6","KLF9","KYNU","LAMB3","LDLR","LIF","LITAF","MAFF","MAP2K3","MAP3K8","MARCKS","MCL1","MSC","MXD1","MYC","NAMPT","NFAT5","NFE2L2","NFIL3","NFKB1","NFKB2","NFKBIA","NFKBIE","NINJ1","NR4A1","NR4A2","NR4A3","OLR1","PANX1","PDE4B","PDLIM5","PER1","PFKFB3","PHLDA1","PHLDA2","PLAU","PLAUR","PLEK","PLK2","PLPP3","PMEPA1","PNRC1","PPP1R15A","PTGER4","PTGS2","PTPRE","PTX3","RCAN1","REL","RELA","RELB","RHOB","RIPK2","RNF19B","SAT1","SDC4","SERPINB2","SERPINB8","SERPINE1","SGK1","SIK1","SLC16A6","SLC2A3","SLC2A6","SMAD3","SNN","SOCS3","SOD2","SPHK1","SPSB1","SQSTM1","STAT5A","TANK","TAP1","TGIF1","TIPARP","TLR2","TNC","TNF","TNFAIP2","TNFAIP3","TNFAIP6","TNFAIP8","TNFRSF9","TNFSF9","TNIP1","TNIP2","TRAF1","TRIB1","TRIP10","TSC22D1","TUBB2A","VEGFA","YRDC","ZBTB10","ZC3H12A","ZFP36"]

JAK_genes = ["A2M","ACVR1B","ACVRL1","BAK1","CBL","CCL7","CCR1","CD14","CD36","CD38","CD44","CD9","CNTFR","CRLF2","CSF1","CSF2","CSF2RA","CSF2RB","CSF3R","CXCL1","CXCL10","CXCL11","CXCL13","CXCL3","CXCL9","DNTT","EBI3","FAS","GRB2","HAX1","HMOX1","IFNAR1","IFNGR1","IFNGR2","IL10RB","IL12RB1","IL13RA1","IL15RA","IL17RA","IL17RB","IL18R1","IL1B","IL1R1","IL1R2","IL2RA","IL2RG","IL3RA","IL4R","IL6","IL6ST","IL7","IL9R","INHBE","IRF1","IRF9","ITGA4","ITGB3","JUN","LEPR","LTB","LTBR","MAP3K8","MYD88","OSMR","PDGFC","PF4","PIK3R5","PIM1","PLA2G2A","PTPN1","PTPN11","PTPN2","REG1A","SOCS1","SOCS3","STAM2","STAT1","STAT2","STAT3","TGFB1","TLR2","TNF","TNFRSF12A","TNFRSF1A","TNFRSF1B","TNFRSF21","TYK2"]

IFN_genes = ["ABCE1","ADAR","BST2","CACTIN","CDC37","CNOT7","DCST1","EGR1","FADD","GBP2","HLA-A","HLA-B","HLA-C","HLA-E","HLA-F","HLA-G","HLA-H","HSP90AB1","IFI27","IFI35","IFI6","IFIT1","IFIT2","IFIT3","IFITM1","IFITM2","IFITM3","IFNA1","IFNA10","IFNA13","IFNA14","IFNA16","IFNA17","IFNA2","IFNA21","IFNA4","IFNA5","IFNA6","IFNA7","IFNA8","IFNAR1","IFNAR2","IFNB1","IKBKE","IP6K2","IRAK1","IRF1","IRF2","IRF3","IRF4","IRF5","IRF6","IRF7","IRF8","IRF9","ISG15","ISG20","JAK1","LSM14A","MAVS","METTL3","MIR21","MMP12","MUL1","MX1","MX2","MYD88","NLRC5","OAS1","OAS2","OAS3","OASL","PSMB8","PTPN1","PTPN11","PTPN2","PTPN6","RNASEL","RSAD2","SAMHD1","SETD2","SHFL","SHMT2","SP100","STAT1","STAT2","TBK1","TREX1","TRIM56","TRIM6","TTLL12","TYK2","UBE2K","USP18","WNT5A","XAF1","YTHDF2","YTHDF3","ZBP1"]

IFNG_genes = ["NR1H3","ACTR3","ACTR2","CDC42EP2","SYNCRIP","ADAMTS13","CDC37","VPS26B","GBP4","GBP5","OTOP1","SIRPA","DAPK1","DAPK3","GBP6","EDN1","AIF1","PDE12","EPRS1","FCAR","FLNB","RPL13A","CDC42EP4","STXBP4","GAPDH","GBP1","GBP2","GBP3","GSN","HCK","HLA-DPA1","HPX","RAB7B","RAB43","IRF8","IRGM","IFNG","IFNGR1","IFNGR2","FASLG","IL12B","IL12RB1","AQP4","IRF1","JAK1","JAK2","KIF5B","ARG1","GBP7","LGALS9","CIITA","CD200","MRC1","ASS1","MYO1C","NOS2","PIM1","PARP14","PPARG","MED1","RAB20","PTPN2","RAF1","RPS6KB1","CCL2","CCL3","CCL5","SLC26A6","SP100","STAT1","STX4","STXBP1","STXBP2","STXBP3","CRIPTO","TLR2","TLR3","TLR4","ACTG1","TNF","TP53","TXK","TYK2","ACOD1","NR1H2","VIM","WAS","WNT5A","ZYX","CALM1","CAMK2A","CASP1","PARP9","NLRC5","VAMP4","CLDN1","DNAJA3","VAMP3","STX8","CD47","CD58","CDC42"]

#TGFB_genes = ["CDH3","CDH5","TGFBR3L","SPRY1","SPRY2","STUB1","CDKN1C","CDKN2B","CITED2","TAB1","LEFTY1","GIPC1","FERMT2","EMILIN1","STRAP","IL17F","CIDEA","VASN","LRG1","WFIKKN1","WFIKKN2","COL1A2","COL3A1","CD109","CREB1","CREBBP","PARP1","DAB2","SPRED1","EID2","DAND5","SPRED2","FLCN","ENG","HTRA4","EP300","MECOM","FBN1","FBN2","FKBP1A","SNW1","PEG10","FMOD","FNTA","SIRT1","FOLR1","FOS","LEMD3","FAM89B","FSHB","FUT8","NPNT","BAMBI","BRMS1","SIN3A","TSKU","ZNF451","APPL1","LRRC32","GCNT2","LATS2","MSTN","GDF9","GDF10","AMHR2","DKK3","GLG1","GOT1","BCL9L","RGCC","HIPK2","ERO1A","HDAC1","HDAC2","ONECUT1","HPGD","HSPA1A","HSPA5","HSP90AB1","APOA1","ID1","ING1","ING2","ITGA3","ITGB1","ITGB5","ITGB6","ITGB8","JUN","NRROS","SPRED3","LOX","LTBP1","LTBP2","LTBP3","MIRLET7A1","MIRLET7B","MIRLET7F1","MIRLET7G","MIR101-1","MIR106A","MIR140","MIR142","MIR145","MIR15B","MIR17","MIR18A","MIR181A2","MIR183","MIR199A1","MIR19A","MIR19B1","MIR20A","MIR204","MIR21","MIR212","MIR26A1","MIR27B","MIR29B1","MIR30A","MIR30B","MIR9-1","MIR93","MIR98","SMAD1","SMAD2","SMAD3","SMAD4","ARRB2","SMAD5","SMAD6","SMAD7","SMAD9","MEN1","MIR302B","MIR323A","MIR342","MIR376C","MIR372","MIR373","CITED1","NDP","NODAL","MIR361","MIR424","FURIN","CHST11","ZBTB7A","TRIM33","PDPK1","NLK","ARID4B","PIN1","PML","PPARA","PPARG","IL17RD","RNF111","ASPN","PPM1A","ADISSP","APPL2","FERMT1","INTS9","HTRA1","PSG9","PMEPA1","DUSP22","TWSG1","SMURF1","ZMIZ1","SELENON","MIR490","MIR497","MIR498","MIR520C","PTK2","PTPRK","PXN","OVOL2","SINHCAF","SNX6","ARID4A","RBBP4","RBBP7","BCL9","SDCBP","PRDM16","PBLD","FKBP1C","PALS1","SUDS3","SMURF2","SKI","SKIL","BMP2","SKOR2","BMPR1A","RASL11B","SOX11","SPI1","SRC","STAT3","STK11","ADAM17","MAP3K7","MIR564","ZEB1","TGFB1","TGFB1I1","TGFB2","TGFB3","TGFBR1","TGFBR2","TGFBR3","THBS1","NKX2-1","CLDN5","TP53","WNT1","XBP1","LDLRAD4","ZYX","SAP130","VEPH1","SAP30L","ZNF703","TET1","SLC2A10","GDF5","USP9X","USP9Y","AXIN1","ZMIZ2","SNX25","LTBP4","BRMS1L","OGT","CILP","ITGA8","CAV2","CAV3","ADAM9","SAP30","TSC22D1","FOXH1","ACVR1","PIAS2","MTMR4","LATS1","NREP","MYOCD","ZFYVE9","TGFBRAP1","ACVRL1","HTRA3","LPXN","ONECUT2","GDF15","ADAMTSL2","ZEB2","USP15"]
TGFB_genes = ["CDH3","CDH5","TGFBR3L","DNM3OS","SPRY1","SPRY2","STUB1","CDKN1C","CDKN2B","CITED2","TAB1","POSTN","LEFTY1","GIPC1","FERMT2","EMILIN1","STRAP","IL17F","CIDEA","VASN","LRG1","WFIKKN1","WFIKKN2","COL1A1","COL1A2","COL3A1","COL4A2","CD109","CREB1","CREBBP","CRK","CRKL","PARP1","DAB2","SPRED1","EID2","DLX1","EDN1","DAND5","SPRED2","FLCN","ENG","HTRA4","EP300","MECOM","FBN1","FBN2","FGFR2","FKBP1A","SNW1","PEG10","FMOD","FNTA","SIRT1","FOLR1","FOS","LEMD3","CD2AP","FAM89B","FSHB","ABL1","FUT8","FYN","NPNT","BAMBI","BRMS1","MXRA5","SIN3A","TSKU","ZNF451","APPL1","LRRC32","GCNT2","LATS2","MSTN","GDF9","GDF10","AMHR2","ANKRD1","DKK3","GLG1","GOT1","BCL9L","RGCC","HIPK2","NR3C1","ERO1A","HDAC1","HDAC2","APAF1","ONECUT1","HPGD","HSPA1A","HSPA5","HSP90AB1","APOA1","ID1","IL4","ING1","ING2","ISL1","ITGA3","ITGB1","ITGB5","ITGB6","ITGB8","JUN","NRROS","LIMS1","SPRED3","LOX","LTBP1","LTBP2","LTBP3","MIRLET7A1","MIRLET7B","MIRLET7F1","MIRLET7G","MIR101-1","MIR106A","MIR130A","MIR140","MIR142","MIR145","MIR15B","MIR17","MIR18A","MIR181A2","MIR183","MIR199A1","MIR19A","MIR19B1","MIR20A","MIR204","MIR21","MIR212","MIR26A1","MIR27A","MIR27B","MIR29B1","MIR30A","MIR30B","MIR9-1","MIR93","MIR98","SMAD1","SMAD2","SMAD3","SMAD4","ARRB2","SMAD5","SMAD6","SMAD7","SMAD9","MEF2C","MEN1","KMT2A","MIR302B","MIR323A","MIR342","MIR376C","MIR372","MIR373","CITED1","ZFHX3","NDP","NODAL","DDR2","MIR361","MIR424","FURIN","CHST11","PAK2","ZBTB7A","PDE2A","PDE3A","TRIM33","PDPK1","NLK","WWOX","ARID4B","PENK","PIN1","PML","WNT4","PPARA","PPARG","IL17RD","RNF111","ASPN","PPM1A","ADISSP","APPL2","SOX6","FERMT1","INTS9","MAPK7","HTRA1","PSG9","PMEPA1","DUSP22","TWSG1","SMURF1","ZMIZ1","SELENON","MIR490","MIR497","MIR498","MIR520C","PTK2","EPB41L5","PTPRK","PXN","OVOL2","SINHCAF","SNX6","ACTA2","ARID4A","RBBP4","RBBP7","BCL9","ROCK1","XCL1","SDCBP","PRDM16","PBLD","SFRP1","FKBP1C","SCX","NFKBIZ","PALS1","SUDS3","SMURF2","FNDC4","SKI","SKIL","BMP2","SKOR2","BMPR1A","RASL11B","SNRNP70","SOX5","SOX9","SOX11","SPI1","SRC","ZFP36L1","STAT3","ZFP36L2","STK11","ADAM17","MAP3K7","MIR564","ZEB1","TGFB1","TGFB1I1","TGFB2","TGFB3","TGFBR1","TGFBR2","TGFBR3","THBS1","NKX2-1","CLDN5","TP53","WNT1","WNT2","WNT5A","WNT7A","XBP1","YES1","LDLRAD4","ZYX","SAP130","VEPH1","SAP30L","ZNF703","PDGFD","TET1","WNT10A","SLC2A10","GDF5","USP9X","USP9Y","AXIN1","ZMIZ2","SNX25","LTBP4","BRMS1L","OGT","CILP","ITGA8","CAV1","STK16","CAV2","CAV3","RUNX3","HYAL2","ADAM9","SAP30","CFLAR","TSC22D1","FOXH1","ACVR1","PIAS2","CLDN1","MTMR4","LATS1","PDCD5","NREP","MYOCD","ZFYVE9","TGFBRAP1","ACVRL1","HTRA3","LPXN","ROCK2","ONECUT2","GDF15","ADAMTSL2","ZEB2","USP15"]
#TGFB_genes = ["ACVR1","APC","ARID4B","BCAR3","BMP2","BMPR1A","BMPR2","CDH1","CDK9","CDKN1C","CTNNB1","ENG","FKBP1A","FNTA","FURIN","HDAC1","HIPK2","ID1","ID2","ID3","IFNGR2","JUNB","KLF10","LEFTY2","LTBP2","MAP3K7","NCOR2","NOG","PMEPA1","PPM1A","PPP1CA","PPP1R15A","RAB31","RHOA","SERPINE1","SKI","SKIL","SLC20A1","SMAD1","SMAD3","SMAD6","SMAD7","SMURF1","SMURF2","SPTBN1","TGFB1","TGFBR1","TGIF1","THBS1","TJP1","TRIM33","UBE2D3","WWTR1","XIAP"]

cytokine_genes = ["TANK","CD24","SH2B3","MIR1246","NR1H3","IFNL4","EBI3","ADAM10","LILRB2","TRAIP","ADAR","CCL26","PIAS3","IFITM3","YAP1","HAX1","CEBPA","CIB1","AGPAT1","AGPAT2","CXCL13","IFITM2","SH2B2","CXCR6","RRAGA","TNFSF13B","TRAF3IP2","CCR9","LILRB1","EDAR","GPR75","LILRB5","LILRB4","P3R3URF","LILRA1","LILRB3","LILRA3","LILRA2","CDC37","IRAK3","PADI2","USP18","IL17F","TREX1","CHUK","CARD16","C1QTNF4","CISH","TNFRSF13C","IL22RA2","RFFL","TRIM6","SOCS4","CCR1","CCR3","CCR4","CCR5","CCR6","CCR7","CCR8","ACKR2","CMKLR1","CNTF","CNTFR","IL17RE","OTOP1","IL31RA","CR2","MIB2","MAPK14","CSF1","CSF1R","CSF2","CSF2RA","CSF2RB","CSF3","CSF3R","CSNK2B","IL34","CD300LF","TMC8","DCST1","IL23R","COMMD7","CTSG","PYDC2","CX3CR1","DAB2IP","CREBRF","CYLD","IFNLR1","NLRP6","ZNF675","DOK1","ECM1","EDA","EDN1","EDN2","EGR1","EIF5A","TRIM65","HIPK1","EPO","EPOR","EREG","AKT1","EXT1","F2RL1","F3","ACSL1","PTK2B","FCER1G","FER","CARD8","TRIM32","FOXC1","FBXO21","FOXO3","TTLL12","FLT3","PLCB1","DICER1","SIRT1","CLCF1","LILRA4","IL17RA","PELI3","ACKR1","YTHDF3","SAMHD1","SIN3A","PYDC1","GIGYF2","APPL1","LSM14A","GAS6","STAP1","IL1RAPL2","IL36RN","GFI1","GH1","GHR","IL36B","IL37","IL36A","RABGEF1","STK39","CCR10","IFNL2","IFNL3","IFNL1","XCR1","CXCR3","ANGPT1","GPR17","NKIRAS2","NKIRAS1","GPR35","NLRP2B","GPS2","PYCARD","TBK1","GSTP1","USP25","CNOT7","HCK","HCLS1","ANXA4","HIF1A","UBE2K","HPX","BIRC2","BIRC3","HSPA1A","HSPA1B","XIAP","APOA1","IFNE","STING1","IFI27","PALM3","IFNA1","IFNA2","IFNA4","IFNA5","IFNA6","IFNA7","IFNA8","IFNA10","IFNA14","IFNA16","IFNA17","IFNA21","IFNAR1","IFNAR2","IFNB1","IRGM","IFNG","IFNGR1","IFNGR2","IFNW1","TICAM2","LILRA5","FAS","IKBKB","IL1A","IL1B","IL1R1","IL1RAP","IL1RN","IL2","IL2RA","IL2RB","IL2RG","IL3","IL3RA","IL4","IL4R","IL5","IL5RA","IL6","IL6R","IL6ST","IL7","IL7R","CXCR1","IL9","CXCR2","IL9R","IL10","IL10RA","IL10RB","IL11","IL11RA","IL12A","IL12B","IL12RB1","IL12RB2","IL13","IL13RA1","IL13RA2","IL15","IL15RA","IL16","IL17A","IL18","ILK","INHBA","CXCL10","IRAK1","IRAK2","IRF1","IRF3","IRF5","IRF7","IRS1","JAK1","JAK2","JAK3","SLC27A1","KIT","ARG1","KRAS","KRT8","KRT18","CCL4L2","USP27X","LEP","LEPR","LIFR","LIMS1","IL17REL","SMIM30","LYN","MIRLET7A1","MIRLET7C","MIRLET7E","MIR101-1","MIR125A","MIR125B1","MIR130A","MIR135A1","MIR146A","MIR152","MIR20A","MIR21","MIR24-1","MIR26A1","MIR27A","MIR27B","MIR29B1","MIR34A","MIR98","MIR99A","SMAD4","CCL3L3","CIITA","CXCL9","MMP8","MMP12","MPL","RHEX","MST1R","MT3","MX1","MYD88","NAIP","NFKB1","NFKBIA","NOTCH1","NRDC","OAS1","OAS2","OAS3","TNFRSF11B","OSM","P4HB","PAFAH1B1","DUOX2","IL21R","PRKN","ADIPOR1","IRAK4","CLDN18","YTHDF2","PDGFB","ACKR4","PIAS4","PF4","PIK3R1","PLP2","IL20RA","IL20RB","DUOX1","TREM2","TOLLIP","RBM47","PARP14","PPARG","MED1","OTUD4","IL17RD","TRIM44","WBP1L","PPP2CB","APPL2","TNFRSF19","IL17RB","PRKACA","AXL","ZC3H15","MAPK1","MAPK3","PRL","PRLR","IL36G","METTL3","IFNK","GPR108","SPPL2B","ACKR3","BAD","MIR520C","MAVS","USP29","PTPN1","PTPN2","EPG5","PTPN6","PTPN11","PTPRC","PTPRJ","BBS2","BBS4","CACTIN","RAF1","IL22RA1","SIGIRR","RELA","EDA2R","TNFRSF17","ROBO1","CEACAM1","CCL1","CCL2","CCL3","CCL3L3","CCL4","CCL5","CCL7","CCL8","CCL11","CCL13","CCL14","CCL15","CCL16","CCL18","CCL19","CCL21","CCL22","CCL23","CCL24","CCL25","CXCL6","CXCL11","XCL1","CX3CL1","CXCL12","CRLF2","IFIH1","SRSF1","CXCR5","NFKBIZ","AZI2","GREM2","RBM15","SLC1A1","WNK1","SLIT3","SOS1","SP100","SPI1","SRC","STAT1","STAT2","STAT3","STAT4","STAT5A","STAT5B","STAT6","XCL2","SYK","ADAM17","MAP3K7","TFF2","THPO","TMSB4X","TNF","TNFAIP3","TNFRSF1A","TNFRSF1B","TP53","NR2C2","TRAF1","TRAF2","TRAF3","TRAF5","TRAF6","CCR2","TNFRSF4","TXK","TYK2","UGCG","UMOD","NR1H2","VRK2","WNT5A","RNF113A","LRP8","LAPTM5","PXDN","IL1R2","CXCR4","CARD14","TNIP2","LILRA6","BIRC7","MUL1","ADIPOR2","FOSL1","ACTN4","ZBP1","TRIM56","SHARPIN","MKKS","FZD4","CASP1","PLVAP","CCDC3","PARP9","CASP4","ULK1","CASP8","NLRC5","NSMAF","JAGN1","IL1F10","TXNDC17","IL17RC","SPPL2A","IFITM1","TSLP","CAV1","TNFSF11","OASL","SOCS1","CBL","TRADD","TNFRSF25","RIPK1","TNFRSF14","RIPK2","FADD","TNFRSF18","TNFRSF11A","IL18RAP","IL1RL2","IL18R1","SOCS2","CDK5R1","SPHK1","TAX1BP1","CPNE1","H2BC11","RPS6KA4","TNFSF18","NOL3","SOCS3","OTULIN","CCRL2","IL33","DNAJA3","TRIM41","CNOT9","RNF185","IL1RL1","OSMR","CD4","CRLF1","RPS6KA5","NUMBL","MAPKAPK2","TIFA","SLIT2","ADIPOQ","ENTREP1","AIM2","IL27RA","EIF4E2","CD40","CD44","ISG15","IKBKE","CTR9","SOCS5","CD70","ST18","CD74","TBKBP1","HDAC4","SPATA2","OXSR1","NR1H4"]

granu_migrate_genes = ["CD300H","DNM1L","ADAM8","CCL26","VAV3","CXCL13","CCL27","CD300A","TIRAP","JAML","CCR7","CMKLR1","FAM3D","CSF1","CSF1R","CSF3R","IL34","CXADR","MOSPD2","DPEP1","DPP4","EDN1","EDN2","EDN3","PIKFYVE","EMP2","FCER1G","PIP5K1C","DAPK2","IL17RA","FUT4","FUT7","MCOLN2","MSTN","C5AR2","GP2","CXCL17","CXCL3","ANXA1","NCKAP1L","ADGRE2","IL1B","IL1R1","CXCL8","CXCR1","CXCR2","IL17A","CXCL10","ITGA1","ITGA9","ITGB2","CCL4L2","LBP","LGALS3","MIR223","MDK","CD99","CXCL9","MPP1","MYD88","IRAK4","CKLF","PDE4B","IL23A","PECAM1","PF4","PF4V1","PIK3CD","PIK3CG","PLA2G1B","TREM1","PPBP","PPIA","PPIB","MAPK1","MAPK3","CCL28","PRTN3","SLAMF8","CAMK1D","CD177","RTN4","PTGER4","S100A14","PTK2","PREX1","PTPRJ","SELENOK","RAC1","RAC2","RAC3","RARRES2","TRPV4","S100A7","S100A8","S100A9","S100A12","SAA1","CCL1","CCL2","CCL3","CCL4","CCL5","CCL7","CCL8","CCL11","CCL13","CCL16","CCL19","CCL21","CCL22","CCL24","CCL25","CXCL6","CXCL5","XCL1","CX3CL1","PERP","SLAMF1","SRP54","BSG","BST1","SYK","TGFB2","THBS1","THBS4","C1QBP","TNFAIP6","C3AR1","C5AR1","UMOD","VAV1","VEGFA","XG","SCG2","AKIRIN1","EPX","CD99L2","JAM3","JAGN1","IL17RC","GBF1","TNFSF18","MCU","SLIT2","CD74","RIPOR2","WDR1"]

mono_genes = ["KLRC4-KLRK1","MICOS10-NBL1","ADAM8","ADAM10","CCL26","CREB3","CXCL13","RIPK3","EMILIN1","CORO1A","PADI2","JAML","CCR1","CCR5","CCR6","CCR7","CMKLR1","SPNS2","CNN2","CD200R1","CRK","CRKL","DEFB104A","SIRPA","CSF1","CSF1R","IL34","GCSAML","CTSG","TAFA4","CX3CR1","MOSPD2","CYP19A1","DDT","DEFA1","DEFA4","AGER","DUSP1","GPR183","ECM1","S1PR1","EDN2","EDNRB","ANO6","AIF1","EPS8","AKT1","EXT1","PTK2B","KLRK1","LRCH1","FLT1","PLCB1","FOLR2","RPL13A","FPR2","ALOX5","DEFB124","ABL1","FUT4","FUT7","MCOLN2","H2BC1","GCSAM","GAS6","STAP1","GATA3","GBA1","GREM1","MSTN","B4GALT1","ABL2","STK39","GNAI1","CXCR3","GPR15","TMEM102","CXCL17","PYCARD","LRP12","TBX21","ANXA1","HMGB1","AIRE","ICAM1","APOD","APP","IL6","IL6R","CXCR1","CXCR2","IL12A","CXCL10","ITGA4","ITGAL","ITGB3","ITGB7","MIA3","RHOA","GPR15LG","LGALS3","LGALS9","LYN","MIR128-1","MIR146A","MIR24-1","MDK","CD99","MIF","ASCL2","MMP2","MMP14","CD200","MSN","NBL1","NEDD9","NINJ1","CCN3","P2RX4","P4HB","DEFB104B","SERPINE1","F11R","CKLF","PDGFB","ASB2","PECAM1","PF4","PIK3CD","PIK3CG","PLEC","PLG","TREM2","TRPM4","MAPK1","MAPK3","CRTAM","LGMN","AZU1","SLAMF8","RTN4","S100A14","PTK2","MTUS1","PTPRJ","PTPRO","CXCL16","JAM2","SELENOK","RARRES2","TRPV4","RET","BCR","RPS19","S100A7","S100A12","SAA1","CCL1","CCL2","CCL3","CCL4","CCL5","CCL7","CCL19","CCL20","CCL21","CCL23","CXCL11","XCL1","CX3CL1","CXCL12","MYO1G","SFTPD","DEFB131A","P2RY12","SLAMF1","WNK1","BMP5","SLC12A2","SPI1","SPN","STK10","ADAM17","MSMP","TGFB1","THBS1","C1QBP","TNF","C3AR1","TRPM2","C5","C5AR1","DEFA1B","CCR2","WNT5A","XG","ZAP70","CXCR4","MMP28","PLA2G7","CALCA","AKIRIN1","ARHGEF5","NUP85","SLC8B1","HSD3B7","PDGFD","CALR","DOCK8","MADCAM1","CD99L2","JAM3","TRIM55","ADTRP","TNFSF11","TNFSF14","TNFRSF14","FADD","TNFRSF11A","WASL","TNFSF18","CH25H","ARTN","NLRP12","CD9","SLIT2","CYP7B1","MED23","IL27RA","CD47","CD69","CD81","RIPOR2","OXSR1","CDC42"]

mhc_genes = ["LILRB2","CLEC4M","NOD1","IFI30","FGL2","RAB10","RAB32","RAB35","TREX1","RAB3C","CCR7","FAM3D","RAET1E","CTSD","CTSE","CTSH","CTSL","CTSV","CTSS","RAET1L","ACE","DNM2","PIKFYVE","EXT1","MPEG1","FCER1G","FCER2","FCGR1A","MARCHF8","FCGR2B","RFTN1","ABCB9","GBA1","PYCARD","PDIA3","HFE","CD209","HLA-A","HLA-B","HLA-C","HLA-DMA","HLA-DMB","HLA-DOA","HLA-DOB","HLA-DPA1","HLA-DPB1","HLA-DQA1","HLA-DQA2","HLA-DQB1","HLA-DQB2","HLA-DRA","HLA-DRB1","HLA-DRB3","HLA-DRB4","HLA-DRB5","HLA-E","HLA-F","HLA-G","HLA-H","MR1","ICAM1","IDE","IGHE","RAET1G","IKBKB","LNPEP","CLEC4A","SAR1B","ERAP1","RAB8B","TREM2","MFSD6","YTHDF1","MARCHF1","TAPBPL","ARL8B","AZGP1","LGMN","B2M","PSMB8","PSME1","WDFY4","RAB3B","RAB4A","RAB5B","RAB6A","RAB27A","RELB","CCL19","CCL21","NOD2","ERAP2","SLC11A1","TAP1","TAP2","TAPBP","THBS1","TRAF6","WAS","ULBP3","ULBP2","ULBP1","CALR","UNC93B1","KDM5D","RAB34","CST7","AP3B1","CTSF","AP3D1","CD1A","CD1B","CD1C","CD1D","CD1E","CD8A","RAB33A","ATG5","CD68","CD74"]
 

In [5]:
# =============================================================================
# Module scoring with scanpy's sc.tl.score_genes (single-cell)
# -> score CELLS per module (scanpy control-gene matching)
# -> pseudobulk by DONOR: mean(cell scores) within ann_lv2 × Condition2 × sampleID
# -> stats: MWU/Wilcoxon rank-sum on DONOR means (CFB vs C, CFN vs C) per ann_lv2
# -> display: 3 heatmaps (IFN/JAK/TNF)
#    - color = mean donor score (optionally z-scored per module for display)
#    - stars = significance (donor-level MWU vs C)
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import scanpy as sc

# -------------------------
# Settings
# -------------------------
CLUSTER_KEY   = "ann_lv2"
CONDITION_KEY = "Condition2"
DONOR_KEY     = "sampleID"

cond_colors = {"C": "#26547c", "CFN": "#ffd166", "CFB": "#ef476f"}

baseline    = "C"
cond_order  = ["C", "CFB", "CFN"]
comparisons = ["CFB", "CFN", "CFB_vs_CFN"]

exclude_lv2 = {}

levels = [
    "PBMC_CD16.mono",
    "PBMC_CD14.mono",
    "PBMC_cDC",
    "BAL_CD14.mono",
    "BAL_cDC",
    "BAL_MDM",
    "BAL_MDM-PLA2G7hi",
    "BAL_AM",
    "BAL_AM-CCL",
]

celltype_preferred = levels

modules = ["TNF", "IFNG","MHC",#"IFN", 
           "JAK", "TGFB",
           "cytokine","granulocyte_migration","mono"
           #"adhesion"
          ]
module_to_gene_list = {"TNF": TNF_genes, "JAK": JAK_genes, "MHC": mhc_genes,#"IFN": IFN_genes,
                      "IFNG": IFNG_genes, "TGFB": TGFB_genes,
                       "cytokine": cytokine_genes,
                       "granulocyte_migration": granu_migrate_genes,
                       "mono": mono_genes
                       #"adhesion":adhesion_genes
                      }
module_title = {"TNF": "TNF response", 
                "IFNG": "Type II IFN response",
                #"IFN": "Type I IFN response",
                "MHC": "Antigen presentation",
                "JAK": "JAK–STAT response",
                "TGFB": r"TGF$\beta$ response",
                "cytokine": "Cytokine signaling",
                "granulocyte_migration": "Granulocyte migration",
                "mono": "Mononuclear cell migration"
                #"adhesion":"Integrin-mediated cell adhesion"
               }

# Make adata.X point to log1p values (WORKING matrix)
adata.X = adata.layers["log1p"].copy()
print("log1p matrix shape:", adata.layers["log1p"].shape)


log1p matrix shape: (218407, 32566)


### Scoring

In [6]:
SCOREGENES_CTRL_SIZE = 50
SCOREGENES_N_BINS = 25
SCOREGENES_USE_RAW = False

# -------------------------
# Helpers
# -------------------------
def stars_from_q(q):
    if pd.isna(q): return ""
    if q < 1e-3: return "***"
    if q < 1e-2: return "**"
    if q < 5e-2: return "*"
    return ""

# -------------------------
# Calculate Module Scores for Each Dataset
# -------------------------
print(f"\nProcessing...")
for mod in modules:
    genes = [g for g in module_to_gene_list[mod] if g in adata.var_names]
    if len(genes) < 5:
        print(f"{mod}: too few genes present for scoring (present={len(genes)}). Skipping.")
        continue
    sc.tl.score_genes(
        adata,
        gene_list=genes,
        score_name=f"{mod}_score",
        ctrl_size=len(genes), #SCOREGENES_CTRL_SIZE,
        n_bins=SCOREGENES_N_BINS,
        use_raw=SCOREGENES_USE_RAW,
        random_state=0,
    )



Processing...
computing score 'TNF_score'
    finished: added
    'TNF_score', score of gene set (adata.obs).
    3163 total control genes are used. (0:00:06)
computing score 'IFNG_score'
    finished: added
    'IFNG_score', score of gene set (adata.obs).
    1891 total control genes are used. (0:00:05)
computing score 'MHC_score'
    finished: added
    'MHC_score', score of gene set (adata.obs).
    2252 total control genes are used. (0:00:05)
computing score 'JAK_score'
    finished: added
    'JAK_score', score of gene set (adata.obs).
    1627 total control genes are used. (0:00:05)
computing score 'TGFB_score'
    finished: added
    'TGFB_score', score of gene set (adata.obs).
    6277 total control genes are used. (0:00:07)
computing score 'cytokine_score'
    finished: added
    'cytokine_score', score of gene set (adata.obs).
    12205 total control genes are used. (0:00:08)
computing score 'granulocyte_migration_score'
    finished: added
    'granulocyte_migration_score',

### Subset + rename

In [7]:
# Assumes:
# - adata is the concatenated object
# - adata.obs["dataset"] is "BAL" or "PBMC"
# - CLUSTER_KEY is your lv2 annotation column

lv2 = adata.obs[CLUSTER_KEY].astype(str)
ds  = adata.obs["dataset"].astype(str)

# -----------------------------
# 1) Subset (on the merged adata)
# -----------------------------
keep_pbmc = (ds == "PBMC") & lv2.isin({"CD14.mono", "CD16.mono","cDC","pDC"})
keep_bal = (ds == "BAL") & (
    lv2.eq("CD14.mono") |
    lv2.str.startswith("AM") |
    lv2.str.startswith("MDM") |
    lv2.eq("cDC") |
    lv2.eq("pDC") |
    lv2.eq("DC.migratory")
)

adata_sub = adata[keep_pbmc | keep_bal].copy()

# -----------------------------
# 2) Rename (still produces PBMC_/BAL_ prefixes)
# -----------------------------
lv2_sub = adata_sub.obs[CLUSTER_KEY].astype(str)

def rename_lv2(x: str) -> str:
    if x == "CD14.mono":
        return "Mono"
    if x == "CD16.mono":
        return "Mono"
    if x == "cDC":
        return "cDC"
    if x == "pDC":
        return "pDC"
    # keep BAL macrophage buckets like your intent
    if x == "AM-CCL":
        return "AM-CCL"
    if x.startswith("AM"):
        return "AM"
    if x == "MDM-PLA2G7hi":
        return "MDM-PLA2G7hi"
    if x == "DC.migratory":
        return "DC.migratory"
    if x.startswith("MDM"):
        return "MDM"
    return np.nan

adata_sub.obs[CLUSTER_KEY] = pd.Categorical(lv2_sub.map(rename_lv2))
adata_sub = adata_sub[adata_sub.obs[CLUSTER_KEY].notna()].copy()

# -----------------------------
# 3) Print counts (by dataset + label)
# -----------------------------
print("Subset counts by dataset:")
print(adata_sub.obs["dataset"].value_counts())

print("\nSubset counts by renamed label:")
print(adata_sub.obs[CLUSTER_KEY].value_counts())

print("\nCross-tab:")
print(pd.crosstab(adata_sub.obs["dataset"], adata_sub.obs[CLUSTER_KEY]))


Subset counts by dataset:
dataset
BAL     86157
PBMC     2257
Name: count, dtype: int64

Subset counts by renamed label:
ann_lv2
AM              69183
Mono             4395
AM-CCL           4177
MDM              4022
cDC              2470
MDM-PLA2G7hi     2273
pDC              1325
DC.migratory      569
Name: count, dtype: int64

Cross-tab:
ann_lv2     AM  AM-CCL  DC.migratory   MDM  MDM-PLA2G7hi  Mono   cDC   pDC
dataset                                                                   
PBMC         0       0             0     0             0  1786   351   120
BAL      69183    4177           569  4022          2273  2609  2119  1205


### Pseudobulk scores

In [8]:
# -------------------------
# Pseudobulk: Mean Scores by Donor
# -------------------------
donor_scores = (
    adata_sub.obs
    .assign(
        ann_lv2=lambda d: d['dataset'].astype(str) + "_" + d[CLUSTER_KEY].astype(str),
        Condition2=lambda d: d[CONDITION_KEY].astype(str),
        sampleID=lambda d: d[DONOR_KEY].astype(str),
    )
    .groupby(["ann_lv2", "Condition2", "sampleID"], dropna=False)
    [[f"{m}_score" for m in modules]]
    .mean()
    .reset_index()
)
donor_scores

,ann_lv2,Condition2,sampleID,TNF_score,IFNG_score,MHC_score,JAK_score,TGFB_score,cytokine_score,granulocyte_migration_score,mono_score
0,BAL_AM,C,M1N066,0.149447,0.380670,0.583803,0.159547,0.133761,0.136483,0.207267,0.187592
1,BAL_AM,C,M1N075,0.163393,0.370582,0.559670,0.146513,0.142896,0.131812,0.213489,0.192269
2,BAL_AM,C,M1N078,0.156436,0.360770,0.535952,0.146807,0.129729,0.125687,0.196965,0.178745
3,BAL_AM,C,M1N080,0.218941,0.424736,0.613718,0.181491,0.158082,0.148366,0.239642,0.213292
4,BAL_AM,C,M1N087,0.166111,0.386928,0.549680,0.159265,0.146871,0.138361,0.221823,0.200611
...,...,...,...,...,...,...,...,...,...,...,...
188,PBMC_pDC,CFN,M1C176C,0.082435,0.191458,0.372197,0.149943,0.167961,0.126030,0.134582,0.136039
189,PBMC_pDC,CFN,M1C180D,0.176549,0.241628,0.377119,0.180048,0.149699,0.103549,0.113146,0.128823
190,PBMC_pDC,CFN,M1C199B,0.137939,0.222247,0.401568,0.143440,0.133418,0.112394,0.143417,0.164994
191,PBMC_pDC,CFN,M1C201A,0.118295,0.202211,0.405072,0.133325,0.144145,0.110730,0.140138,0.150781


In [9]:
donor_scores["ann_lv2"].value_counts().head(50)

ann_lv2
BAL_AM              18
BAL_AM-CCL          18
BAL_DC.migratory    18
BAL_MDM             18
BAL_MDM-PLA2G7hi    18
BAL_Mono            18
BAL_cDC             18
BAL_pDC             18
PBMC_Mono           17
PBMC_cDC            17
PBMC_pDC            15
Name: count, dtype: int64

In [10]:
# -------------------------
# Statistical Testing: MWU/Wilcoxon rank-sum (donor means)
# Single concatenated donor_scores (no dataset split):
#   loop: ann_lv2 (celltype) × module × pairwise comparison
#   BH correction within each (ann_lv2 × module) panel across the 3 pairwise p-values
# -------------------------

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

PAIRWISE = [
    ("CFB", "C",   "CFB vs C",   "CFB"),
    ("CFN", "C",   "CFN vs C",   "CFN"),
    ("CFB", "CFN", "CFB vs CFN", "CFB_vs_CFN"),
]

# modules should be like: ["TNF","IFNG","MHC","JAK","TGFB","cytokine","granulocyte_migration","mono"]
# and columns in donor_scores must be e.g. TNF_score, IFNG_score, ...

required_cols = {"ann_lv2", "Condition2", "sampleID"}
missing = required_cols - set(donor_scores.columns)
if missing:
    raise ValueError(f"donor_scores is missing required columns: {sorted(missing)}")

# ensure types
d = donor_scores.copy()
d["ann_lv2"] = d["ann_lv2"].astype(str)
d["Condition2"] = d["Condition2"].astype(str)
d["sampleID"] = d["sampleID"].astype(str)

stat_rows = []

for ct in d["ann_lv2"].dropna().unique():
    d_ct = d[d["ann_lv2"] == ct]

    for mod in modules:
        col = f"{mod}_score"
        if col not in d_ct.columns:
            continue

        for g1, g2, label, key in PAIRWISE:
            x = d_ct.loc[d_ct["Condition2"] == g1, col].astype(float).dropna().values
            y = d_ct.loc[d_ct["Condition2"] == g2, col].astype(float).dropna().values

            if (len(x) < 2) or (len(y) < 2):
                p = np.nan
            else:
                p = mannwhitneyu(x, y, alternative="two-sided").pvalue

            stat_rows.append({
                "ann_lv2": ct,
                "module": mod,
                "comparison": label,   # human label
                "Condition2": key,     # key used for star_map / plotting
                "group1": g1,
                "group2": g2,
                "p": p,
                "n_group1_donors": int(len(x)),
                "n_group2_donors": int(len(y)),
            })

stats_grp = pd.DataFrame(stat_rows)
stats_grp["q"] = np.nan

# -------------------------
# BH adjust within each panel (ann_lv2 × module) across the 3 pairwise p-values
# -------------------------
for (ct, mod), sub_idx in stats_grp.groupby(["ann_lv2", "module"]).groups.items():
    sub_idx = list(sub_idx)
    pvals = stats_grp.loc[sub_idx, "p"].values
    m = ~np.isnan(pvals)
    if m.sum() == 0:
        continue

    _, qvals, *_ = multipletests(pvals[m], method="fdr_bh")
    q_full = np.full_like(pvals, np.nan, dtype=float)
    q_full[m] = qvals
    stats_grp.loc[sub_idx, "q"] = q_full

def stars_from_q(q):
    if pd.isna(q): return ""
    if q < 1e-3: return "***"
    if q < 1e-2: return "**"
    if q < 5e-2: return "*"
    return ""

stats_grp["stars"] = stats_grp["q"].map(stars_from_q)

# star_map includes all 3 comparisons
star_map = {
    (r.ann_lv2, r.module, r.Condition2): (r.stars if isinstance(r.stars, str) else "")
    for r in stats_grp.itertuples(index=False)
}

# -------------------------
# Print significant results
# -------------------------
sig = stats_grp[stats_grp["q"].notna() & (stats_grp["q"] < 0.05)].copy()
if len(sig):
    print(
        sig.sort_values(["comparison", "q", "ann_lv2", "module"])[
            ["ann_lv2","module","comparison","p","q","stars","n_group1_donors","n_group2_donors"]
        ].to_string(index=False)
    )
else:
    print("No significant results at q < 0.05 (panel-wise BH).")

         ann_lv2                module comparison        p        q stars  n_group1_donors  n_group2_donors
BAL_DC.migratory                   MHC   CFN vs C 0.002165 0.006494    **                6                6
BAL_MDM-PLA2G7hi granulocyte_migration   CFN vs C 0.008658 0.025974     *                6                6
         BAL_pDC                   MHC   CFN vs C 0.008658 0.025974     *                6                6


NameError: name 'disp_long' is not defined

In [17]:
# -------------------------
# Define celltypes from both subsets
# -------------------------
celltypes = ["PBMC_Mono","BAL_Mono","BAL_MDM"]

# -------------------------
# Heatmap Values: Mean of Donor Means (per subtype × condition)
# -------------------------
score_cols = [f"{m}_score" for m in modules]

display_means = (
    donor_scores
    .groupby(["ann_lv2", "Condition2"], dropna=False)[score_cols]
    .mean()
    .reset_index()
)

disp_long = display_means.melt(
    id_vars=["ann_lv2", "Condition2"],
    value_vars=score_cols,
    var_name="module_score",
    value_name="score",
)
disp_long["module"] = disp_long["module_score"].str.replace("_score", "", regex=False)

# Add dataset column (match your established heatmap chunk)
disp_long["dataset"] = np.where(
    disp_long["ann_lv2"].astype(str).str.startswith("PBMC_"), "PBMC",
    np.where(disp_long["ann_lv2"].astype(str).str.startswith("BAL_"), "BAL", "PBMC")
)

# Optional: z-score per module across all subtypes/conditions (for visualization)
disp_long["score_z"] = np.nan
for mod in modules:
    mm = disp_long["module"].eq(mod)
    v = disp_long.loc[mm, "score"].astype(float).values
    mu = np.nanmean(v)
    sd = np.nanstd(v)
    disp_long.loc[mm, "score_z"] = (disp_long.loc[mm, "score"].astype(float) - mu) / (sd + 1e-8)

# -------------------------
# Plot: Heatmaps grouped by Cell Type (One Per Cell Type)
# -------------------------
n_celltypes = len(celltypes)
n_cols = 3
n_rows = int(np.ceil(n_celltypes / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6, 2.5 * n_rows), sharey=True)
axes = axes.ravel()
fig.subplots_adjust(hspace=0.45, top=0.92)

cmap = plt.get_cmap("PuOr_r")
vmin, vmax = -2, 2

x_edges = np.arange(len(cond_order) + 1)
y_edges = np.arange(len(modules) + 1)
x_centers = np.arange(len(cond_order)) + 0.5
y_centers = np.arange(len(modules)) + 0.5

mappable = None

for ax_idx, ct in enumerate(celltypes):
    ax = axes[ax_idx]

    # Decide dataset exactly like your established chunk
    if str(ct).startswith("PBMC_"):
        ds = "PBMC"
    elif str(ct).startswith("BAL_"):
        ds = "BAL"
    else:
        ds = "PBMC"

    subct = disp_long[(disp_long["dataset"] == ds) & (disp_long["ann_lv2"] == ct)].copy()

    # Build matrix: rows = modules, columns = conditions
    d = {(r.module, r.Condition2): r.score_z for r in subct.itertuples(index=False)}
    Z = np.full((len(modules), len(cond_order)), np.nan, dtype=float)
    for rr, mod in enumerate(modules):
        for cc, cond in enumerate(cond_order):
            Z[rr, cc] = d.get((mod, cond), np.nan)

    pm = ax.pcolormesh(x_edges, y_edges, Z, cmap=cmap, vmin=vmin, vmax=vmax, shading="flat")
    if mappable is None:
        mappable = pm

    # Match established formatting
    ax.set_xticks(np.arange(len(cond_order) + 1), minor=True)
    ax.set_yticks(np.arange(len(modules) + 1), minor=True)
    ax.grid(which="major", color="white", linewidth=0.2)
    ax.tick_params(which="major", bottom=False, left=False)

    ax.set_title(ct, fontsize=9, fontweight="normal")
    ax.set_xlim(0, len(cond_order))
    ax.set_ylim(0, len(modules))

    ax.set_xticks(x_centers)
    ax.set_xticklabels(cond_order, rotation=0)

    ax.set_yticks(y_centers)
    ax.set_yticklabels([module_title[m] for m in modules])

    # Stars on non-baseline cells only (vs baseline)
    # Match established star_map signature: (ds, ct, mod, cond)
    for rr, mod in enumerate(modules):
        for cc, cond in enumerate(cond_order):
            if cond == baseline:
                continue
            st = star_map.get((ds, ct, mod, cond), "")
            if st:
                ax.text(cc + 0.5, rr + 0.5, st, ha="center", va="center",
                        fontsize=8, color="black")

# Hide unused subplots
for ax_idx in range(n_celltypes, len(axes)):
    axes[ax_idx].axis("off")

# Colorbar: match established settings
cbar = fig.colorbar(
    mappable,
    ax=axes.ravel().tolist(),
    fraction=0.04,
    pad=0.2,
    location="right",
    shrink=0.3,
    anchor=(4.0, 0.5),
    panchor=(4.0, 0.5),
    aspect=10
)
cbar.set_label("Module z-score")

plt.tight_layout()
plt.show()

# out = os.path.join(OUTDIR, "ModuleScore_MWU.jpeg")
# fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")
# print("Saved:", out)

/tmp/ipykernel_2729634/3918079237.py:131: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


NameError: name 'out' is not defined

## Paired t test

### PBMC Mono vs BAL Mono

In [14]:
import pandas as pd
from pathlib import Path

# Keep only these cell types
keep_cell_types = ["PBMC_Mono", "BAL_Mono"]

# Filter first
df_filt = donor_scores[donor_scores["ann_lv2"].isin(keep_cell_types)].copy()

# Long format
id_cols = ["ann_lv2", "Condition2", "sampleID"]
long_df = (
    df_filt.melt(
        id_vars=id_cols,
        var_name="Pathway",
        value_name="Score",
    )
    .rename(columns={"ann_lv2": "Cell type"})
)

# Optional: remove trailing "_score"
long_df["Pathway"] = long_df["Pathway"].str.replace(r"_score$", "", regex=True)

# Exact column order
long_df = long_df[["Cell type", "Condition2", "sampleID", "Pathway", "Score"]]

# Export
out_dir = Path("/group/canc2/anson/working/cf-pbmc-bal/data")
out_path = out_dir / "myeloid_donor_scores.xlsx"
out_dir.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    long_df.to_excel(writer, sheet_name="long_scores", index=False)

print(f"Saved: {out_path}")

Saved: /group/canc2/anson/working/cf-pbmc-bal/data/myeloid_donor_scores.xlsx


In [ ]:
# Optimized + flexible (works with 3 CTs or 4 CTs)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests

alpha = 0.05
min_donors = 2

assert "donor_scores" in globals(), "donor_scores must exist."
assert "modules" in globals()
conds_to_plot = [ "CFB", "CFN"] # "C",
cond_color = {"C": "#26547c", "CFN": "#ffd166", "CFB": "#ef476f"}
assert "stars_from_q" in globals()

# -----------------------------
# 1) Configure CTs flexibly
# -----------------------------
# Put ANY CTs you want here (3 or 4 or more). Adjacent comparisons are tested/plotted.          # example: 3 CTs
ct_list = ["Mono"]
for ct in ct_list:
    cts = [f"PBMC_{ct}",f"BAL_{ct}"]
    
    # Labels must match length of cts
    ct_label_map = {
        f"PBMC_{ct}":  f"PBMC\n{ct}",
        f"BAL_{ct}":  f"BAL\n{ct}",
    }
    x_labels = [ct_label_map.get(c, c) for c in cts]
    
    # Auto x positions (even spacing)
    x_span = 0.2   # was effectively ~0.66; smaller => CTs closer together
    x_ticks = np.linspace(0.0, x_span, num=len(cts))
    x_map = dict(zip(cts, x_ticks))
    x_pad = 0.06
    
    # Adjacent comparisons (families)
    adj_pairs = list(zip(cts[:-1], cts[1:]))
    
    # Optional per-condition rule: skip first pair for "C" (like your original)
    def pairs_for_condition(cond):
        if cond == "C":
            return adj_pairs[1:]  # skip (cts[0] -> cts[1])
        return adj_pairs
    
    # -----------------------------
    # 2) Precompute long form once
    # -----------------------------
    score_cols = [f"{m}_score" for m in modules]
    plot_long = donor_scores.melt(
        id_vars=["ann_lv2", "Condition2", "sampleID"],
        value_vars=score_cols,
        var_name="module_score",
        value_name="score",
    )
    plot_long["module"] = plot_long["module_score"].str.replace("_score", "", regex=False)
    
    # -----------------------------
    # 3) Precompute wide pivots ONCE per (module, condition)
    # -----------------------------
    # pivots[(module, cond)] = wide df: index=sampleID, columns=ann_lv2, values=score
    pivots = {}
    for mod in modules:
        dmod = plot_long[plot_long["module"].eq(mod)]
        for cond in conds_to_plot:
            d = dmod[dmod["Condition2"].eq(cond)]
            piv = d.pivot_table(
                index="sampleID", columns="ann_lv2", values="score",
                aggfunc="mean", observed=False
            )
            # ensure requested CT columns exist (in correct order), missing => NaN
            piv = piv.reindex(columns=cts)
            pivots[(mod, cond)] = piv
    
    # -----------------------------
    # 4) Compute all adjacent paired tests (BH per comparison family)
    # -----------------------------
    rows = []
    for mod in modules:
        for cond in conds_to_plot:
            piv = pivots[(mod, cond)]
            for (ctA, ctB) in adj_pairs:
                piv2 = piv.dropna(subset=[ctA, ctB], how="any")
                n = int(piv2.shape[0])
                if n < 2:
                    t, p = np.nan, np.nan
                else:
                    t, p = ttest_rel(piv2[ctA].to_numpy(), piv2[ctB].to_numpy(), nan_policy="omit")
                rows.append({
                    "Condition2": cond, "module": mod,
                    "ctA": ctA, "ctB": ctB,
                    "comparison": f"{ctA} vs {ctB}",
                    "n_donors": n, "t": t, "p": p
                })
    
    stats_all = pd.DataFrame(rows)
    
    # BH adjust *within each adjacent-pair family* (ctA,ctB), across all (cond,module)
    stats_all["p_adj"] = np.nan
    for (ctA, ctB), subidx in stats_all.groupby(["ctA", "ctB"]).groups.items():
        m = stats_all.loc[subidx, "p"].notna()
        if m.sum() > 0:
            stats_all.loc[subidx[m], "p_adj"] = multipletests(stats_all.loc[subidx[m], "p"], method="fdr_bh")[1]
    
    stats_all["stars"] = stats_all["p_adj"].map(stars_from_q)
    
    # Fast lookup: (cond, mod, ctA, ctB) -> (p_adj, stars, n)
    pmap = {
        (r.Condition2, r.module, r.ctA, r.ctB): (r.p_adj, r.stars, r.n_donors)
        for r in stats_all.itertuples(index=False)
    }
    
    # -----------------------------
    # 5) Plot
    # -----------------------------
    outdir = os.path.join(ROOT, "data", "plots")
    os.makedirs(outdir, exist_ok=True)
    
    for mod in modules:
        # -----------------------------
        # y-limits from ACTUAL plotted data (selected CTs + selected conditions),
        # plus extra headroom for significance bars/text.
        # -----------------------------
        vals_for_ylim = []
        for cond in conds_to_plot:
            piv = pivots[(mod, cond)].copy()
            if piv.shape[0] == 0:
                continue
            n_present = piv.notna().sum(axis=1)
            piv_plot = piv.loc[n_present >= 2]   # same rule as plotting
            if piv_plot.shape[0] == 0:
                continue
            vals_for_ylim.append(piv_plot.to_numpy().ravel())
        
        if len(vals_for_ylim) == 0:
            y_min, y_max = -0.1, 0.1
        else:
            vv = np.concatenate(vals_for_ylim)
            vv = vv[np.isfinite(vv)]
            if vv.size == 0:
                y_min, y_max = -0.1, 0.1
            else:
                # use strict min/max (or swap to quantiles if you prefer robustness)
                y_min, y_max = float(np.min(vv)), float(np.max(vv))
        
        data_range = (y_max - y_min) if (y_max > y_min) else 1.0
        
        # how many bracket rows could appear at most in a panel?
        # (you only draw adjacent pairs; you then filter to significant ones, but reserve worst-case space)
        max_brackets = max(len(pairs_for_condition(c)) for c in conds_to_plot) if conds_to_plot else 0
        
        # spacing must match your annotation spacing so the reserved headroom is accurate
        spacing = 0.06 * data_range
        tick_h = spacing * 0.15
        text_pad = spacing * 0.35  # extra for the star text above the line
        
        top_headroom = max_brackets * spacing + tick_h + text_pad
        bottom_pad = 0.08 * data_range
        
        y_lim = (y_min - bottom_pad, y_max + top_headroom)
    
        fig, axes = plt.subplots(1, len(conds_to_plot), figsize=(2 * len(conds_to_plot), 2.5), sharey=True)
        axes = np.atleast_1d(axes)
        fig.subplots_adjust(wspace=0.25, top=0.86)
    
        for ax, cond in zip(axes, conds_to_plot):
            piv = pivots[(mod, cond)].copy()
    
            if piv.shape[0] == 0:
                ax.text(0.5, 0.5, "no donors", transform=ax.transAxes, ha="center", va="center")
                ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, fontsize=9)
                ax.set_title(cond, fontsize=11); ax.set_ylim(*y_lim)
                continue
    
            # donors with >=2 present points among chosen CTs
            n_present = piv.notna().sum(axis=1)
            piv_plot = piv.loc[n_present >= 2].sort_index()
    
            col = cond_color.get(cond, "black")
            for donor, row in piv_plot.iterrows():
                ys_all = row.to_numpy()
                mask = ~np.isnan(ys_all)
                xs = x_ticks[mask]
                ys = ys_all[mask]
                ax.plot(xs, ys, color="0.75", linewidth=1.0, zorder=1)
                ax.scatter(xs, ys, s=22, color=col, edgecolor="black", linewidth=0.3, zorder=2)
    
            ax.set_title(cond, fontsize=11)
            ax.set_xticks(x_ticks)
            ax.set_xticklabels(x_labels, fontsize=9)
            ax.set_xlim(np.min(x_ticks) - x_pad, np.max(x_ticks) + x_pad)
            ax.set_ylim(*y_lim)
            import matplotlib.ticker as mtick
            ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.2f'))
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
    
            # --- Significant annotations (only for allowed pairs for this condition) ---
            y_top = y_lim[1]
            # stack only the significant ones, so gaps don't appear when one comparison isn't significant
            sig_pairs = []
            for (ctA, ctB) in pairs_for_condition(cond):
                p_adj, stars, n_d = pmap.get((cond, mod, ctA, ctB), (np.nan, "", 0))
                if (n_d is not None) and (n_d >= min_donors) and (pd.notna(p_adj)) and (p_adj < alpha):
                    sig_pairs.append((ctA, ctB, p_adj, stars, n_d))
    
            spacing = 0.06 * data_range
            for j, (ctA, ctB, p_adj, stars, n_d) in enumerate(sig_pairs):
                x1, x2 = x_map[ctA], x_map[ctB]
                y_line = y_top - (j + 1) * spacing
                tick_h = spacing * 0.15
    
                ax.plot([x1, x2], [y_line, y_line], color="k", linewidth=1.0, zorder=3)
                ax.plot([x1, x1], [y_line, y_line - tick_h], color="k", linewidth=1.0, zorder=3)
                ax.plot([x2, x2], [y_line, y_line - tick_h], color="k", linewidth=1.0, zorder=3)
    
                star_text = stars.strip() if isinstance(stars, str) else ""
                #txt = f"{star_text} p.adj={p_adj:.3f}".strip()
                txt = f"{star_text}".strip()
                ax.text((x1 + x2) / 2, y_line + spacing * 0.12, txt,
                        ha="center", va="bottom", fontsize=8, zorder=4)
    
        axes[0].set_ylabel("Mean module score")
        fig.suptitle(module_title.get(mod, mod), fontsize=13, y=0.98)
    
        fig.tight_layout()
        # os.makedirs(os.path.join(OUTDIR,ct), exist_ok=True)
        out = os.path.join(OUTDIR, ct, f"pairedT_{mod}_PBMC-{ct}_vs_BAL-{ct}_forCS.jpeg")
        # fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")
        #plt.show()
        print("Saved:", out)
    
    # -----------------------------
    # 6) Optional: show significant rows
    # -----------------------------
    def show_significant(stats_df, alpha=0.05, min_donors=2):
        sig = stats_df[(stats_df["p_adj"].notna()) & (stats_df["p_adj"] < alpha) & (stats_df["n_donors"] >= min_donors)]
        if sig.empty:
            print(f"No significant results (p_adj < {alpha}).")
            return
        cols = ["Condition2", "module", "comparison", "n_donors", "t", "p", "p_adj", "stars"]
        display(sig.loc[:, cols].sort_values(["Condition2", "module", "comparison"]).reset_index(drop=True))
    
    show_significant(stats_all, alpha=alpha, min_donors=min_donors)

In [19]:
from pathlib import Path
import pandas as pd

# out_dir: Path (as you defined)
# stats_all: DataFrame created above

assert "stats_all" in globals(), "stats_all must exist (paired t-test results DataFrame)."
assert "out_dir" in globals(), "out_dir must exist (Path)."

out_dir = Path(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "myeloid_paired_ttest_statistics.xlsx"

# Optional: nicer ordering / columns
cols = [c for c in ["Condition2", "module", "ctA", "ctB", "comparison", "n_donors", "t", "p", "p_adj", "stars"] if c in stats_all.columns]
stats_export = stats_all.loc[:, cols].sort_values(["Condition2", "module", "ctA", "ctB"]).reset_index(drop=True)

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    stats_export.to_excel(writer, sheet_name="paired_ttests", index=False)

print(f"Saved: {out_path}")

Saved: /group/canc2/anson/working/cf-pbmc-bal/data/myeloid_paired_ttest_statistics.xlsx


### BAL MDM vs BAL AM

In [11]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import multipletests

alpha = 0.05
min_donors = 2

assert "donor_scores" in globals(), "donor_scores must exist."
assert "modules" in globals()
assert "stars_from_q" in globals()

conds_to_plot = ["CFB", "CFN"]
cond_color = {"C": "#26547c", "CFN": "#ffd166", "CFB": "#ef476f"}

# -----------------------------
# 1) Configure CTs for BAL_MDM vs BAL_AM
# -----------------------------
cts = ["BAL_MDM", "BAL_AM"]   # must match donor_scores["ann_lv2"] values exactly

ct_label_map = {
    "BAL_MDM": "BAL\nMDM",
    "BAL_AM":  "BAL\nAM",
}
x_labels = [ct_label_map.get(c, c) for c in cts]

x_span = 0.2
x_ticks = np.linspace(0.0, x_span, num=len(cts))
x_map = dict(zip(cts, x_ticks))
x_pad = 0.06

# only one adjacent comparison here
adj_pairs = list(zip(cts[:-1], cts[1:]))

def pairs_for_condition(cond):
    # keep as-is (hook for special rules)
    return adj_pairs

# -----------------------------
# 2) Precompute long form once
# -----------------------------
score_cols = [f"{m}_score" for m in modules]
plot_long = donor_scores.melt(
    id_vars=["ann_lv2", "Condition2", "sampleID"],
    value_vars=score_cols,
    var_name="module_score",
    value_name="score",
)
plot_long["module"] = plot_long["module_score"].str.replace("_score", "", regex=False)

# -----------------------------
# 3) Precompute wide pivots ONCE per (module, condition)
# -----------------------------
pivots = {}
for mod in modules:
    dmod = plot_long[plot_long["module"].eq(mod)]
    for cond in conds_to_plot:
        d = dmod[dmod["Condition2"].eq(cond)]
        piv = d.pivot_table(
            index="sampleID", columns="ann_lv2", values="score",
            aggfunc="mean", observed=False
        )
        piv = piv.reindex(columns=cts)  # ensure both columns exist
        pivots[(mod, cond)] = piv

# -----------------------------
# 4) Compute all paired tests (BH within pair family)
# -----------------------------
rows = []
for mod in modules:
    for cond in conds_to_plot:
        piv = pivots[(mod, cond)]
        for (ctA, ctB) in adj_pairs:
            piv2 = piv.dropna(subset=[ctA, ctB], how="any")
            n = int(piv2.shape[0])
            if n < 2:
                t, p = np.nan, np.nan
            else:
                t, p = ttest_rel(piv2[ctA].to_numpy(), piv2[ctB].to_numpy(), nan_policy="omit")
            rows.append({
                "Condition2": cond, "module": mod,
                "ctA": ctA, "ctB": ctB,
                "comparison": f"{ctA} vs {ctB}",
                "n_donors": n, "t": t, "p": p
            })

stats_all = pd.DataFrame(rows)

stats_all["p_adj"] = np.nan
for (ctA, ctB), subidx in stats_all.groupby(["ctA", "ctB"]).groups.items():
    m = stats_all.loc[subidx, "p"].notna()
    if m.sum() > 0:
        stats_all.loc[subidx[m], "p_adj"] = multipletests(stats_all.loc[subidx[m], "p"], method="fdr_bh")[1]

stats_all["stars"] = stats_all["p_adj"].map(stars_from_q)

pmap = {
    (r.Condition2, r.module, r.ctA, r.ctB): (r.p_adj, r.stars, r.n_donors)
    for r in stats_all.itertuples(index=False)
}

# -----------------------------
# 5) Plot
# -----------------------------
for mod in modules:
    # y-limits based on actual plotted data
    vals_for_ylim = []
    for cond in conds_to_plot:
        piv = pivots[(mod, cond)].copy()
        if piv.shape[0] == 0:
            continue
        n_present = piv.notna().sum(axis=1)
        piv_plot = piv.loc[n_present >= 2]  # must have both BAL_MDM and BAL_AM
        if piv_plot.shape[0] == 0:
            continue
        vals_for_ylim.append(piv_plot.to_numpy().ravel())

    if len(vals_for_ylim) == 0:
        y_min, y_max = -0.1, 0.1
    else:
        vv = np.concatenate(vals_for_ylim)
        vv = vv[np.isfinite(vv)]
        y_min, y_max = (float(np.min(vv)), float(np.max(vv))) if vv.size else (-0.1, 0.1)

    data_range = (y_max - y_min) if (y_max > y_min) else 1.0
    max_brackets = max(len(pairs_for_condition(c)) for c in conds_to_plot) if conds_to_plot else 0
    spacing = 0.06 * data_range
    tick_h = spacing * 0.15
    text_pad = spacing * 0.35
    top_headroom = max_brackets * spacing + tick_h + text_pad
    bottom_pad = 0.08 * data_range
    y_lim = (y_min - bottom_pad, y_max + top_headroom)

    fig, axes = plt.subplots(1, len(conds_to_plot), figsize=(2 * len(conds_to_plot), 2.5), sharey=True)
    axes = np.atleast_1d(axes)
    fig.subplots_adjust(wspace=0.25, top=0.86)

    for ax, cond in zip(axes, conds_to_plot):
        piv = pivots[(mod, cond)].copy()

        if piv.shape[0] == 0:
            ax.text(0.5, 0.5, "no donors", transform=ax.transAxes, ha="center", va="center")
            ax.set_xticks(x_ticks); ax.set_xticklabels(x_labels, fontsize=9)
            ax.set_title(cond, fontsize=11); ax.set_ylim(*y_lim)
            continue

        piv_plot = piv.dropna(subset=cts, how="any").sort_index()  # require paired points

        col = cond_color.get(cond, "black")
        for donor, row in piv_plot.iterrows():
            ys_all = row.to_numpy()
            xs = x_ticks
            ys = ys_all
            ax.plot(xs, ys, color="0.75", linewidth=1.0, zorder=1)
            ax.scatter(xs, ys, s=22, color=col, edgecolor="black", linewidth=0.3, zorder=2)

        ax.set_title(cond, fontsize=11)
        ax.set_xticks(x_ticks)
        ax.set_xticklabels(x_labels, fontsize=9)
        ax.set_xlim(np.min(x_ticks) - x_pad, np.max(x_ticks) + x_pad)
        ax.set_ylim(*y_lim)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        # Significant annotation (only one pair here)
        y_top = y_lim[1]
        sig_pairs = []
        for (ctA, ctB) in pairs_for_condition(cond):
            p_adj, stars, n_d = pmap.get((cond, mod, ctA, ctB), (np.nan, "", 0))
            if (n_d is not None) and (n_d >= min_donors) and (pd.notna(p_adj)) and (p_adj < alpha):
                sig_pairs.append((ctA, ctB, p_adj, stars, n_d))

        for j, (ctA, ctB, p_adj, stars, n_d) in enumerate(sig_pairs):
            x1, x2 = x_map[ctA], x_map[ctB]
            y_line = y_top - (j + 1) * spacing

            ax.plot([x1, x2], [y_line, y_line], color="k", linewidth=1.0, zorder=3)
            ax.plot([x1, x1], [y_line, y_line - tick_h], color="k", linewidth=1.0, zorder=3)
            ax.plot([x2, x2], [y_line, y_line - tick_h], color="k", linewidth=1.0, zorder=3)

            star_text = stars.strip() if isinstance(stars, str) else ""
            ax.text((x1 + x2) / 2, y_line + spacing * 0.12, star_text,
                    ha="center", va="bottom", fontsize=8, zorder=4)

    axes[0].set_ylabel("Mean module score")
    fig.suptitle(module_title.get(mod, mod), fontsize=13, y=0.98)

    fig.tight_layout()
    os.makedirs(OUTDIR, exist_ok=True)
    out = os.path.join(OUTDIR, f"pairedT_{mod}_BAL_MDM_vs_BAL_AM.jpeg")
    fig.savefig(out, dpi=1600, format="jpeg", bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print("Saved:", out)

# -----------------------------
# 6) Optional: show significant rows
# -----------------------------
def show_significant(stats_df, alpha=0.05, min_donors=2):
    sig = stats_df[(stats_df["p_adj"].notna()) & (stats_df["p_adj"] < alpha) & (stats_df["n_donors"] >= min_donors)]
    if sig.empty:
        print(f"No significant results (p_adj < {alpha}).")
        return
    cols = ["Condition2", "module", "comparison", "n_donors", "t", "p", "p_adj", "stars"]
    display(sig.loc[:, cols].sort_values(["Condition2", "module", "comparison"]).reset_index(drop=True))

show_significant(stats_all, alpha=alpha, min_donors=min_donors)

Saved: /group/canc2/anson/working/cf-pbmc-bal/data/plots/module_scoring/pairedT_TNF_BAL_MDM_vs_BAL_AM.jpeg
Saved: /group/canc2/anson/working/cf-pbmc-bal/data/plots/module_scoring/pairedT_IFNG_BAL_MDM_vs_BAL_AM.jpeg
Saved: /group/canc2/anson/working/cf-pbmc-bal/data/plots/module_scoring/pairedT_MHC_BAL_MDM_vs_BAL_AM.jpeg
Saved: /group/canc2/anson/working/cf-pbmc-bal/data/plots/module_scoring/pairedT_JAK_BAL_MDM_vs_BAL_AM.jpeg
Saved: /group/canc2/anson/working/cf-pbmc-bal/data/plots/module_scoring/pairedT_TGFB_BAL_MDM_vs_BAL_AM.jpeg
Saved: /group/canc2/anson/working/cf-pbmc-bal/data/plots/module_scoring/pairedT_cytokine_BAL_MDM_vs_BAL_AM.jpeg
Saved: /group/canc2/anson/working/cf-pbmc-bal/data/plots/module_scoring/pairedT_granulocyte_migration_BAL_MDM_vs_BAL_AM.jpeg
Saved: /group/canc2/anson/working/cf-pbmc-bal/data/plots/module_scoring/pairedT_mono_BAL_MDM_vs_BAL_AM.jpeg


,Condition2,module,comparison,n_donors,t,p,p_adj,stars
0,CFB,IFNG,BAL_MDM vs BAL_AM,6,4.727685,0.005207,0.014285,*
1,CFB,JAK,BAL_MDM vs BAL_AM,6,4.556317,0.006077,0.014285,*
2,CFB,TGFB,BAL_MDM vs BAL_AM,6,4.525778,0.006250,0.014285,*
3,CFB,TNF,BAL_MDM vs BAL_AM,6,3.407180,0.019102,0.023510,*
4,CFB,cytokine,BAL_MDM vs BAL_AM,6,4.966397,0.004225,0.014285,*
5,CFB,granulocyte_migration,BAL_MDM vs BAL_AM,6,4.006052,0.010262,0.018243,*
6,CFB,mono,BAL_MDM vs BAL_AM,6,3.631157,0.015042,0.020056,*
7,CFN,IFNG,BAL_MDM vs BAL_AM,6,4.902565,0.004465,0.014285,*
8,CFN,JAK,BAL_MDM vs BAL_AM,6,4.819619,0.004800,0.014285,*
9,CFN,TGFB,BAL_MDM vs BAL_AM,6,2.788705,0.038504,0.044004,*
